# Style Matcher – Software Golden Model (Python)

This notebook implements the **YCbCr-based style matcher** using standard Python/Numpy/OpenCV utilities (no fixed-point quantization, no histogram subsampling).

## 0. Setup

In [57]:
import numpy as np
import cv2
import rawpy
import time

from pathlib import Path

ref_idx = 0

## 1. DNG → RGB (RAW processing)

Using **rawpy** to do demosaic + camera white balance in the RAW/linear domain, then apply gamma for display.

In [58]:
def dng_to_rgb8(in_path: str) -> np.ndarray:
    """Read a DNG file and return an 8-bit RGB image (H, W, 3), dtype=uint8."""
    with rawpy.imread(in_path) as raw:
        # 取得 Bayer RAW（線性域）
        bayer   = raw.raw_image.astype(np.float32)
        pattern = raw.raw_pattern           # e.g. [[0 1],[3 2]]
        colors  = raw.color_desc.decode()   # e.g. 'RGBG'
        H, W = bayer.shape

        # ---- Let rawpy do WB + demosaic in RAW/linear domain, then gamma ----
        rgb = raw.postprocess(
            output_bps=8,              # output 8-bit for convenience
            no_auto_bright=True,       # don't auto-brighten
            gamma=(2.2, 4.5),          # sRGB-like gamma
            use_camera_wb=True         # use camera WB
        )
    # rawpy returns RGB by default
    return rgb

## 2. RGB ↔ YCbCr (BT.601 full-range)

We keep the same math as the hardware version, but implemented in float.

- Input/Output images are **BGR** (OpenCV order) for convenience.
- Y, Cb, Cr are in the full-range [0, 255].

In [59]:
def bgr_to_ycbcr(img_bgr: np.ndarray) -> np.ndarray:
    """BGR uint8 -> YCbCr float32, BT.601 full range."""
    assert img_bgr.dtype == np.uint8 and img_bgr.ndim == 3 and img_bgr.shape[2] == 3
    B = img_bgr[..., 0].astype(np.float32)
    G = img_bgr[..., 1].astype(np.float32)
    R = img_bgr[..., 2].astype(np.float32)

    # BT.601 full-range
    Y  =  0.299   * R + 0.587   * G + 0.114   * B
    Cb = -0.168736* R - 0.331264* G + 0.5     * B + 128.0
    Cr =  0.5     * R - 0.418688* G - 0.081312* B + 128.0

    ycbcr = np.stack([Y, Cb, Cr], axis=-1)
    return np.clip(ycbcr, 0.0, 255.0).astype(np.float32)


def ycbcr_to_bgr(ycbcr: np.ndarray) -> np.ndarray:
    """YCbCr float32 (full-range) -> BGR uint8."""
    assert ycbcr.ndim == 3 and ycbcr.shape[2] == 3
    Y  = ycbcr[..., 0].astype(np.float32)
    Cb = ycbcr[..., 1].astype(np.float32) - 128.0
    Cr = ycbcr[..., 2].astype(np.float32) - 128.0

    # Inverse BT.601 full-range
    R = Y + 1.402    * Cr
    G = Y - 0.344136 * Cb - 0.714136 * Cr
    B = Y + 1.772    * Cb

    bgr = np.stack([B, G, R], axis=-1)
    bgr = np.clip(bgr, 0.0, 255.0).astype(np.uint8)
    return bgr

## 3. Y histogram matching (256 bins, full image)

We compute full 256-bin histograms and CDFs with `np.bincount`, then build a LUT by CDF matching.

In [60]:
def compute_y_lut(Y_src_u8: np.ndarray, Y_ref_u8: np.ndarray) -> np.ndarray:
    """Compute LUT[0..255] such that src Y histogram matches ref Y histogram (CDF matching)."""
    assert Y_src_u8.dtype == np.uint8 and Y_ref_u8.dtype == np.uint8

    hist_src = np.bincount(Y_src_u8.ravel(), minlength=256).astype(np.float64)
    hist_ref = np.bincount(Y_ref_u8.ravel(), minlength=256).astype(np.float64)

    cdf_src = np.cumsum(hist_src)
    cdf_ref = np.cumsum(hist_ref)
    cdf_src /= cdf_src[-1]
    cdf_ref /= cdf_ref[-1]

    # For each src level g, find smallest h with cdf_ref[h] >= cdf_src[g]
    lut = np.searchsorted(cdf_ref, cdf_src, side='left').astype(np.uint8)
    return lut


def apply_lut_u8(img_u8: np.ndarray, lut_u8: np.ndarray) -> np.ndarray:
    assert img_u8.dtype == np.uint8 and lut_u8.shape == (256,)
    return lut_u8[img_u8]

## 4. Chroma mean/std matching (Cb/Cr)

Simple 1D Gaussian transfer:

\[ C' = (C - \mu_s) \cdot \frac{\sigma_r}{\sigma_s} + \mu_r \]

with a small epsilon to avoid division-by-zero.

In [61]:
def chroma_mean_std(ycbcr: np.ndarray):
    """Return (mu_Cb, std_Cb, mu_Cr, std_Cr) for float YCbCr image."""
    Cb = ycbcr[..., 1].astype(np.float32)
    Cr = ycbcr[..., 2].astype(np.float32)

    mu_cb  = float(np.mean(Cb))
    std_cb = float(np.std(Cb))
    mu_cr  = float(np.mean(Cr))
    std_cr = float(np.std(Cr))
    return mu_cb, std_cb, mu_cr, std_cr


def apply_chroma_transfer(ycbcr_src: np.ndarray, stats_src, stats_ref, eps: float = 1e-6) -> np.ndarray:
    """Match Cb/Cr mean & std of src to ref (float32 in full-range [0,255])."""
    mu_cb_s, std_cb_s, mu_cr_s, std_cr_s = stats_src
    mu_cb_r, std_cb_r, mu_cr_r, std_cr_r = stats_ref

    out = ycbcr_src.astype(np.float32).copy()
    Cb = out[..., 1]
    Cr = out[..., 2]

    gain_cb = std_cb_r / max(std_cb_s, eps)
    gain_cr = std_cr_r / max(std_cr_s, eps)

    out[..., 1] = (Cb - mu_cb_s) * gain_cb + mu_cb_r
    out[..., 2] = (Cr - mu_cr_s) * gain_cr + mu_cr_r

    out[..., 1:] = np.clip(out[..., 1:], 0.0, 255.0)
    return out

## 5. Full color transfer pipeline (YCbCr)

1) Convert to YCbCr
2) Histogram match **Y**
3) Mean/std match **Cb/Cr**
4) Convert back to BGR

In [62]:
def color_transfer_ycbcr(src_bgr_u8: np.ndarray, ref_bgr_u8: np.ndarray) -> np.ndarray:
    src_ycbcr = bgr_to_ycbcr(src_bgr_u8)
    ref_ycbcr = bgr_to_ycbcr(ref_bgr_u8)

    # --- Y histogram matching ---
    Y_src = src_ycbcr[..., 0].astype(np.uint8)
    Y_ref = ref_ycbcr[..., 0].astype(np.uint8)
    lut_y = compute_y_lut(Y_src, Y_ref)

    src_ycbcr2 = src_ycbcr.copy()
    src_ycbcr2[..., 0] = apply_lut_u8(Y_src, lut_y).astype(np.float32)

    # --- Cb/Cr mean-std matching ---
    stats_src = chroma_mean_std(src_ycbcr2)
    stats_ref = chroma_mean_std(ref_ycbcr)
    src_ycbcr3 = apply_chroma_transfer(src_ycbcr2, stats_src, stats_ref)

    out_bgr = ycbcr_to_bgr(src_ycbcr3)
    return out_bgr

## 6. Demo / batch run

- `ref_path`: a reference style image (e.g., JPG/PNG)
- `src_path`: a source DNG (or a regular image for quick tests)

This block handles both cases.

In [63]:
def read_image_bgr(path: str) -> np.ndarray:
    """Read an image into BGR uint8. Supports common images and DNG."""
    p = Path(path)
    if p.suffix.lower() in {'.dng', '.nef', '.cr2', '.arw', '.rw2'}:
        rgb = dng_to_rgb8(str(p))
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        return bgr
    else:
        bgr = cv2.imread(str(p), cv2.IMREAD_COLOR)
        if bgr is None:
            raise FileNotFoundError(f"Cannot read image: {path}")
        return bgr

In [ ]:
def int_exp(num, bit=2):
    num = str(num)
    while (len(num) < bit):
        num = "0" + num
    return num

total_time = 0

for _ in range(30):
    ref_path = f"TP/ref/a{ref_idx+1}.jpg"      # example reference
    src_path = f"TP/src/pat_{int_exp(_)}.dng"   # example source

    start = time.time_ns()
    # Run
    ref_bgr = read_image_bgr(ref_path)
    src_bgr = read_image_bgr(src_path)

    out_bgr = color_transfer_ycbcr(src_bgr, ref_bgr)

    end = time.time_ns()

    total_time += end - start

    out_path = f"TP/output/output{int_exp(_)}.png"
    cv2.imwrite(out_path, out_bgr)
    print("Saved:", out_path)

print(f"Timing: {round(total_time / 30 / 10**6, 2)} ms/frame")

Saved: TP/output/output00.png
Saved: TP/output/output01.png
Saved: TP/output/output02.png
Saved: TP/output/output03.png
Saved: TP/output/output04.png
Saved: TP/output/output05.png
Saved: TP/output/output06.png
Saved: TP/output/output07.png
Saved: TP/output/output08.png
Saved: TP/output/output09.png
Saved: TP/output/output10.png
Saved: TP/output/output11.png
Saved: TP/output/output12.png
Saved: TP/output/output13.png
Saved: TP/output/output14.png
Saved: TP/output/output15.png
Saved: TP/output/output16.png
Saved: TP/output/output17.png
Saved: TP/output/output18.png
Saved: TP/output/output19.png


KeyboardInterrupt: 